# ⚙️ 02. Feature Engineering & Preprocessing
**Project**: XGBoost-Powered PE Malware Detection with Interaction Constraints  
**Purpose**: Scaler comparison (RobustScaler vs StandardScaler), handling class imbalance, and defining Domain Interaction Constraints.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path('../').resolve()))
from utils.preprocessing import MalwarePreprocessor, prepare_splits, compute_scale_pos_weight, get_interaction_constraints, ALL_FEATURES

data_path = Path('../data/synthetic_malware_data.csv')
df = pd.read_csv(data_path)
print(f"Loaded dataset: {df.shape}")


## 1. Train / Validation / Test Splitting (Stratified)
We split data into 70% Train, 15% Validation, and 15% Test with stratification to preserve the malware ratio.


In [ ]:
train_df, val_df, test_df = prepare_splits(df, train_ratio=0.70, val_ratio=0.15, test_ratio=0.15, random_seed=42)

print(f"Train set: {train_df.shape} (Malware: {train_df['label'].mean():.2%})")
print(f"Val set:   {val_df.shape} (Malware: {val_df['label'].mean():.2%})")
print(f"Test set:  {test_df.shape} (Malware: {test_df['label'].mean():.2%})")


## 2. Preprocessing & Outlier-Robust Scaling
Malware file sizes and API call frequencies exhibit high skewness and heavy tails. `RobustScaler` uses median and IQR to prevent extreme outliers from distorting gradients.


In [ ]:
preprocessor = MalwarePreprocessor(scaler_type='robust')
X_train = preprocessor.fit_transform(train_df)
X_val = preprocessor.transform(val_df)
X_test = preprocessor.transform(test_df)

y_train = train_df['label'].values
y_val = val_df['label'].values
y_test = test_df['label'].values

print(f"Processed X_train shape: {X_train.shape}")


## 3. Class Imbalance Weight Calculation
`scale_pos_weight` is set to `(n_negative / n_positive)` to penalize false negatives in gradient boosting.


In [ ]:
spw = compute_scale_pos_weight(train_df['label'])
print(f"Calculated scale_pos_weight: {spw:.3f}")


## 4. Interaction Constraints Definition
We enforce domain-specific feature grouping in XGBoost to prevent spurious cross-domain correlations (e.g., file size interacting with network IP count).


In [ ]:
constraints = get_interaction_constraints(ALL_FEATURES)
print(f"Number of constraint groups: {len(constraints)}")
for i, group in enumerate(constraints):
    names = [ALL_FEATURES[idx] for idx in group]
    print(f"Group {i+1} ({len(group)} features): {', '.join(names[:4])}{'...' if len(names) > 4 else ''}")
